# Exercises XP: Vector Databases and RAG
Use this guided notebook and fill each TODO before running cells.

## What you'll learn
- Vector search strategies (KNN, ANN) and evaluation.
- Vector database utility (similarity search, RAG).
- Differences between vector DBs, libraries, and plugins.
- Best practices for vector store usage and performance.
- How LMs use context; embedding generation and storage.
- Querying vector stores and applying LMs for QA with retrieved context.

## What you'll build
A functional RAG pipeline with FAISS and ChromaDB, plus QA over retrieved context using a Hugging Face model.

## 0. Setup
Run the install cell once. If your platform needs system deps (e.g., libomp for FAISS), follow instructions in comments.

In [ ]:
%pip uninstall -y pydantic-core pydantic
%pip install -U "pydantic<2"
%pip install -U "faiss-cpu>=1.8.0" "chromadb==0.3.21"
%pip install -U "numpy<2" sentence-transformers transformers

In [ ]:
import os
import json
from pathlib import Path
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from IPython.display import display

os.makedirs('cache', exist_ok=True)

## 🌟 Exercise 1 · Data loading and preparation

In [ ]:
# Load the dataset (semicolon-separated)
data_path = 'labelled_newscatcher_dataset.csv'
pdf = pd.read_csv(data_path, sep=';')

# Add an 'id' column if it doesn't already exist
if 'id' not in pdf.columns:
    pdf['id'] = range(len(pdf))

# Inspect the full dataset
display(pdf.head())
print(f"Shape: {pdf.shape}")
print(f"Columns: {pdf.columns.tolist()}")
print(f"\nMissing values:\n{pdf.isnull().sum()}")

# Create a manageable subset (first 1000 rows)
pdf_subset = pdf.head(1000).copy()

print(f"\nSubset shape: {pdf_subset.shape}")
pdf_subset[['id', 'title']].head()

## 🌟 Exercise 2 · Vectorization with Sentence Transformers

In [ ]:
def example_create_fn(idx: int, text: str) -> InputExample:
    """
    Helper function that creates a sentence_transformer InputExample
    with a guid (string ID), a text (the news title), and a dummy label.
    """
    return InputExample(guid=str(idx), texts=[text], label=0.0)


# Create training examples from the subset using the helper function
faiss_train_examples = [
    example_create_fn(row['id'], row['title'])
    for _, row in pdf_subset.iterrows()
]

# Preview the first 2 examples
for ex in faiss_train_examples[:2]:
    print(f"guid={ex.guid} | text={ex.texts[0][:80]}")

In [ ]:
# Initialize the Sentence Transformer embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Extract the titles as a list of strings
titles_list = pdf_subset['title'].tolist()

# Generate embeddings for all titles
faiss_title_embedding = model.encode(titles_list, convert_to_numpy=True, show_progress_bar=True)

# Check dimensions: (number of articles, vector size)
print(f"Number of embeddings: {len(faiss_title_embedding)}")
print(f"Embedding dimension:  {len(faiss_title_embedding[0])}")
# each article is now represented as a dense 384-dimensional vector
# (all-MiniLM-L6-v2 produces 384-dim vectors)

## 🌟 Exercise 3 · FAISS indexing and search

In [ ]:
# The subset DataFrame that will be searched
pdf_to_index = pdf_subset.copy()

# Integer array of IDs — required by FAISS IndexIDMap
id_index = pdf_to_index['id'].to_numpy().astype(np.int64)

# Copy and cast embeddings to float32 (FAISS requirement)
content_encoded_normalized = faiss_title_embedding.astype('float32').copy()

# Normalize to unit length → cosine similarity == inner product
faiss.normalize_L2(content_encoded_normalized)

# Build the index:
# - IndexFlatIP: brute-force inner product (= cosine sim for normalized vecs)
# - IndexIDMap: wraps it so search returns our original IDs
embedding_dim = content_encoded_normalized.shape[1]  # 384
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(embedding_dim))
index_content.add_with_ids(content_encoded_normalized, id_index)

print(f"Vectors indexed: {index_content.ntotal}")

In [ ]:
def search_content(query: str, pdf_to_index: pd.DataFrame, k: int = 3) -> pd.DataFrame:
    """
    Encode a query string, search the FAISS index, and return the
    top-k most similar articles with their cosine similarity scores.
    """
    # 1. Encode the query into a vector
    query_vector = model.encode([query], convert_to_numpy=True).astype('float32')

    # 2. Normalize for cosine similarity
    faiss.normalize_L2(query_vector)

    # 3. Search: returns (similarities, ids) each of shape (1, k)
    sims, ids = index_content.search(query_vector, k)

    # 4. Retrieve the matching rows from the DataFrame
    results = pdf_to_index[pdf_to_index['id'].isin(ids[0])].copy()

    # 5. Attach similarity scores
    # Build a mapping id → similarity to preserve order after the isin filter
    id_to_sim = dict(zip(ids[0], sims[0]))
    results['similarities'] = results['id'].map(id_to_sim)
    results = results.sort_values('similarities', ascending=False)

    return results[['id', 'title', 'topic', 'similarities']]


# Test the search function
display(search_content('animal', pdf_to_index, k=5))

## 🌟 Exercise 4 · ChromaDB collection and querying

In [ ]:
# Initialize a ChromaDB in-memory client
chroma_client = chromadb.Client(Settings(anonymized_telemetry=False))

collection_name = 'my_news'

# Delete the collection if it already exists (avoids duplicate-ID errors)
if any(c.name == collection_name for c in chroma_client.list_collections()):
    chroma_client.delete_collection(name=collection_name)

print(f"Creating collection: '{collection_name}'")

# Create a fresh collection
# ChromaDB will handle embedding generation automatically
collection = chroma_client.create_collection(name=collection_name)

# Add the first 100 news titles along with their topic metadata
# IDs must be strings in ChromaDB
collection.add(
    documents=pdf_subset['title'][:100].tolist(),
    metadatas=[{'topic': topic} for topic in pdf_subset['topic'][:100].tolist()],
    ids=[str(i) for i in pdf_subset['id'][:100].tolist()]
)

print(f"Documents in collection: {collection.count()}")

# Query the collection: retrieve the 10 most relevant articles for 'space'
results = collection.query(
    query_texts=['space'],
    n_results=10
)

print(json.dumps(results, indent=2))

## 🌟 Exercise 5 · Question answering with a Hugging Face model

In [ ]:
# flan-t5-small is lightweight, instruction-tuned, and well-suited for QA
model_id = 'google/flan-t5-small'

# Load tokenizer and causal language model
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id, ignore_mismatched_sizes=True)

# Build the text-generation pipeline
# flan-t5 is a seq2seq model → use 'text2text-generation'
pipe = pipeline(
    'text2text-generation',
    model=model_id,       # pass the model ID directly so the pipeline
    tokenizer=tokenizer,  # loads the correct architecture automatically
    max_new_tokens=128,
    device_map='auto',
)

# Define the user question
question = "What's the latest news on space development?"

# Use the top-3 documents retrieved by ChromaDB as context
context_docs = results['documents'][0][:3]
context = ' '.join(context_docs)

# Build the RAG prompt
prompt = (
    f"Answer the question using only the context below.\n"
    f"Context: {context}\n"
    f"Question: {question}\n"
    f"Answer:"
)

print("=== PROMPT ===")
print(prompt)
print("\n=== GENERATED ANSWER ===")

# Generate the response
lm_response = pipe(prompt)
print(lm_response[0]['generated_text'])

In [ ]:
# ── Experiment: vary the question and context window size ──────────────────

def ask(question: str, query_term: str, n_docs: int = 5):
    """
    Full RAG loop:
    1. Retrieve `n_docs` relevant articles from ChromaDB using `query_term`.
    2. Build a prompt combining context + question.
    3. Generate and print the answer.
    """
    retrieved = collection.query(query_texts=[query_term], n_results=n_docs)
    context = ' '.join(retrieved['documents'][0])
    prompt = (
        f"Answer the question using only the context below.\n"
        f"Context: {context}\n"
        f"Question: {question}\n"
        f"Answer:"
    )
    answer = pipe(prompt)[0]['generated_text']
    print(f"Q: {question}")
    print(f"A: {answer}\n")


ask("Which companies are involved in satellite technology?", "satellite", n_docs=5)
ask("What are scientists discovering about the universe?", "astronomy", n_docs=3)